In [ ]:
# Paths are relative to the notebooks/ folder of this repository.
# On Google Colab, clone the repository first and run from notebooks/:
#   !git clone https://github.com/Syukrondzeko/natural-disaster-app-review-analysis.git
#   %cd natural-disaster-app-review-analysis/notebooks
DATA_DIR = "../data"

In [68]:
import pandas as pd

path = f"{DATA_DIR}/1_raw/final_raw_df.xlsx"

df = pd.read_excel(path)

source_counts = df['source'].value_counts()
print(source_counts)



source
play_store    45250
app_store     15223
Name: count, dtype: int64


In [69]:
df

,id,app,content,score,source
0,1,disaster alert,Good App,5,play_store
1,2,disaster alert,"Working as a Public Health Nurse, I get to res...",5,play_store
2,3,disaster alert,"Nice to have, before traveling to unknown terr...",5,play_store
3,4,disaster alert,I like! I'm trying to find anything about tsun...,5,play_store
4,5,disaster alert,Not accurate,1,play_store
...,...,...,...,...,...
60468,60469,yurekuru call,やっぱりこちらのナマズ版ゆれくるが一番シンプルで良いですね。 以前と同じだと良いのですが…と...,3,app_store
60469,60470,yurekuru call,『ゆれくる』とても便利だったのに突然違うアプリに変更されて使い勝手が悪く削除しようか迷ってい...,4,app_store
60470,60471,yurekuru call,ようやく再リリースされました。早速改悪アプリは即刻削除しました。今後、昔ながらの実用性に期待...,3,app_store
60471,60472,yurekuru call,やっぱり、ゆれくるが1番使いやすいですね！ まだインストールしてから地震は発生してませんが復...,5,app_store


In [70]:
import pandas as pd

input_path = f"{DATA_DIR}/2_translated/final_translated_df.xlsx"
df = pd.read_excel(input_path)


In [71]:
df

,id,app,content,score,source,language,content_translated
0,1,disaster alert,Good App,5,play_store,English,Good App
1,2,disaster alert,"Working as a Public Health Nurse, I get to res...",5,play_store,English,"Working as a Public Health Nurse, I get to res..."
2,3,disaster alert,"Nice to have, before traveling to unknown terr...",5,play_store,English,"Nice to have, before traveling to unknown terr..."
3,4,disaster alert,I like! I'm trying to find anything about tsun...,5,play_store,English,I like! I'm trying to find anything about tsun...
4,5,disaster alert,Not accurate,1,play_store,English,Not accurate
...,...,...,...,...,...,...,...
60239,33384,my earthquake alerts,খুব বেশি কার্যকর নয়।,5,play_store,Bengali,Not very effective.
60240,33400,my earthquake alerts,"এটি একটি ভালো মাধ্যম, যার মাধ্যমে বিভিন্ন মানু...",3,play_store,Bengali,It is a good way to warn people about the eart...
60241,33409,my earthquake alerts,খুবই ভালো,4,play_store,Bengali,Very good.
60242,33541,my earthquake alerts,ধন্যবাদ সংশ্লিষ্ট সবাইকে,5,play_store,Bengali,Thanks to all the people involved.


In [72]:
english_count = (df['language'] == 'English').sum()
print(english_count)
print(len(df)-english_count)

52297
7947


In [73]:
before_count = df.shape[0]

# Lowercase
df['content_translated'] = df['content_translated'].str.lower()

# Remove comments with less than 4 words
df_clean = df[df['content_translated'].str.split().str.len() >= 4]

after_count = df_clean.shape[0]

print(f"Rows before removal: {before_count}")
print(f"Rows after removal: {after_count}")
print(f"Rows removed: {before_count - after_count}")


Rows before removal: 60244
Rows after removal: 49025
Rows removed: 11219


In [74]:
df_clean

,id,app,content,score,source,language,content_translated
1,2,disaster alert,"Working as a Public Health Nurse, I get to res...",5,play_store,English,"working as a public health nurse, i get to res..."
2,3,disaster alert,"Nice to have, before traveling to unknown terr...",5,play_store,English,"nice to have, before traveling to unknown terr..."
3,4,disaster alert,I like! I'm trying to find anything about tsun...,5,play_store,English,i like! i'm trying to find anything about tsun...
5,6,disaster alert,good to have but what options are expected in ...,5,play_store,English,good to have but what options are expected in ...
6,7,disaster alert,"Shows hazards, all right, but refuses to send ...",2,play_store,English,"shows hazards, all right, but refuses to send ..."
...,...,...,...,...,...,...,...
60234,23153,bd 999,৯৯৯ এপপস এর apk লিংটা দেওয়া যাবে ভাইয়া আমি এক...,5,play_store,Bengali,i'll use the llpe link to your apps and increa...
60237,27858,earthquakes near me,সাথে সাথে পাওয়া যায়না alert,2,play_store,Bengali,you can't get with me
60238,33377,my earthquake alerts,সঠিক লোকেশন দেন না কেন,1,play_store,Bengali,why don't you give me the exact location?
60240,33400,my earthquake alerts,"এটি একটি ভালো মাধ্যম, যার মাধ্যমে বিভিন্ন মানু...",3,play_store,Bengali,it is a good way to warn people about the eart...


In [75]:
import pandas as pd
import unicodedata
import re

before_count = df_clean.shape[0]  # after 4-word filter

# Normalize and clean text
def clean_text(text):
    if pd.isna(text):
        return text
    # Normalize unicode
    text = unicodedata.normalize('NFKC', text)
    # Replace curly quotes with ASCII
    text = text.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")
    # Remove invisible/non-printable characters
    text = "".join(c for c in text if c.isprintable())
    return text

df_clean['content_translated'] = df_clean['content_translated'].apply(clean_text)

# Function to detect non-Latin letters in text
def has_non_latin_word(text):
    words = re.findall(r'\w+', text)
    for word in words:
        for char in word:
            if char.isalpha() and not ('a' <= char.lower() <= 'z'):
                return True
    return False

# Function to detect any emoji in text
def has_emoji(text):
    # Unicode ranges covering all emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"  # Misc symbols and pictographs
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F680-\U0001F6FF"  # Transport & map symbols
        "\U0001F700-\U0001F77F"  # Alchemical symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols, Symbols & Pictographs Extended-A
        "\U0001FA70-\U0001FAFF"  # Symbols & Pictographs Extended-B
        "\U00002702-\U000027B0"  # Dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return bool(emoji_pattern.search(text))

# Mask: keep rows without any non-Latin letters AND without any emoji
latin_mask = ~df_clean['content_translated'].apply(has_non_latin_word) & ~df_clean['content_translated'].apply(has_emoji)

# Rows removed
removed_rows = df_clean[~latin_mask]

# New cleaned dataframe
df_clean_2 = df_clean[latin_mask]

after_count = df_clean_2.shape[0]

print(f"Rows before removal: {before_count}")
print(f"Rows after removal: {after_count}")
print(f"Rows removed: {before_count - after_count}")

# See removed rows
print("\nSample of removed rows:")
print(removed_rows.head(10))


/tmp/ipython-input-3357213290.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['content_translated'] = df_clean['content_translated'].apply(clean_text)


Rows before removal: 49025
Rows after removal: 48621
Rows removed: 404

Sample of removed rows:
        id                app  \
85      89     disaster alert   
188    202     disaster alert   
329    353     disaster alert   
1040  1108  earthquake alert!   
1220  1302  earthquake alert!   
1351  1451  earthquake alert!   
1401  1507  earthquake alert!   
1466  1575  earthquake alert!   
1510  1622  earthquake alert!   
1523  1636  earthquake alert!   

                                                content  score      source  \
85    Nice to have a warning ️ prior it has been int...      5  play_store   
188   This APP is full of BUGS... I had to restart m...      1  play_store   
329                      Safe and Alert!!! You Rock!! ️      5  play_store   
1040  Maryborough Qld has got a sister city and it's...      5  play_store   
1220  Good app my country is 🇮🇳 India i see there is...      5  play_store   
1351  अभी तक कोई अनुभव नहीं हुआ है होगा तो आपको अवश्...      5  play_sto

In [76]:
# Drop the 'id' column
df_clean_2 = df_clean_2.drop(columns=['id'])

# Optional: check the columns
print(df_clean_2.columns)


Index(['app', 'content', 'score', 'source', 'language', 'content_translated'], dtype='object')


In [77]:
# path = f"{DATA_DIR}/3_clean/clean_df.xlsx"

# df_clean_2.to_excel(path, index=False)
